In [49]:
import requests
import re

In [ ]:
SPARQL_ENDPOINT = 'http://localhost:7001'
CRISP_BASE_URI = 'http://crispy.ai.wu.ac.at/crisp/'

In [ ]:
class SogoScaling:

	def run_sparql_query(self, query):
		response = requests.get(SPARQL_ENDPOINT, params={'query': query, 'format': 'json'})
		return response.json()['results']['bindings']


	def SumAggregation(self, parts_dict):
		return sum(parts_dict.values())

	def run_scaling_method(self, function_name, parts):
		func = getattr(self, function_name)
		return func(parts)


	def get_feature_class(self):
		"""
		Queries the knowledge graph to determine the class name of the target region. According to the SOGO specification, this class should be a subclass of geo:Feature.

		Returns:
			URI of the target feature's class.
		"""
		query = f'''
		PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
		PREFIX geo: <http://www.opengis.net/ont/geosparql#>

		SELECT ?target_feature WHERE {{
			<{self.target_region_uri}> a ?target_feature .
			?target_feature rdfs:subClassOf geo:Feature .
		}}
		'''
		result = self.run_sparql_query(query)
		return result[0]['target_feature']['value'] if result else None
	

	def fetch_scaling_procedure(self, target_feature):
		"""
		Fetch the scaling procedure matching the given feature class from knowledge graph.
		TODO: Temporal extent should be parsed properly
		"""

		query = f'''
		PREFIX sogo: <http://example.org/sogo#>
		PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

		SELECT ?procedure ?map ?method ?temporal_extent WHERE {{

		?procedure a sogo:ScalingProcedure ;
			sogo:hasScalingProperty <{self.scaling_property_uri}> ;
			sogo:usesScalingMethod ?method ;
			sogo:hasTemporalExtent ?temporal_extent ;
			sogo:hasSpatialMap ?map .
			?map sogo:targetFeature <{target_feature}> .
		}}'''	

		result = self.run_sparql_query(query)

		if not result:
			print("No scaling procedure found for the given property and target feature.")
			return None, None, None, None


		self.scaling_procedure = result[0]['procedure']['value']
		self.scaling_map = result[0]['map']['value']
		self.scaling_method = result[0]['method']['value'].split('#')[-1]  
		# self.temporal_extent = result[0]['temporal_extent']['value']


	def fetch_mapping_query(self):
		query = f'''
		PREFIX sogo: <http://example.org/sogo#>

		SELECT ?mapping_query WHERE {{
			<{self.scaling_map}> sogo:mappingQuery ?mapping_query .
		}}
		'''
		result = self.run_sparql_query(query)
		if not result:
			return None
		
		mapping_query = result[0]['mapping_query']['value'] 
		select_pos = re.search(r"\bSELECT\b", mapping_query, re.IGNORECASE).start()
		prefixes = mapping_query[:select_pos].strip()
		query = mapping_query[select_pos:].strip()

		return prefixes, query



	def get_observation_values(self):
		"""
		Fetching observation values of the part regions to generate the valu for the target observation(s) 

		Retures:
			Dictionary of observation IDs and their corrsponding values.
		"""

		TYPE_CONVERTERS = {
			"http://www.w3.org/2001/XMLSchema#int": int,
			"http://www.w3.org/2001/XMLSchema#integer": int,
			"http://www.w3.org/2001/XMLSchema#float": float,
			"http://www.w3.org/2001/XMLSchema#double": float,
			"http://www.w3.org/2001/XMLSchema#boolean": lambda x: x.lower() == "true",
			"http://www.w3.org/2001/XMLSchema#string": str,
		}

		# fetching the spatial map's query
		mapping_prefixes, mapping_query = self.fetch_mapping_query()
		mapping_query = mapping_query.replace("{SOGO_WHOLE}", f"<{self.target_region_uri}>")

		query = f'''
		PREFIX sosa:  <http://www.w3.org/ns/sosa/>
		{mapping_prefixes}

		SELECT ?part ?scaling_property_value 
		WHERE {{
			{{
				{mapping_query}
			}}

			?obs sosa:hasFeatureOfInterest ?part ;
				sosa:observedProperty <{self.scaling_property_uri}> ;
				sosa:hasSimpleResult ?scaling_property_value ;
				sosa:resultTime ?resultTime .
				FILTER (YEAR(?resultTime) = {self.temporal_extent})

		}} '''

		results = self.run_sparql_query(query)

		parts_dict = dict()

		for item in results:
			part_uri = item['part']['value']
			value = item['scaling_property_value']['value']
			datatype = item['scaling_property_value']['datatype']
			
			converter = TYPE_CONVERTERS.get(datatype, str)
			converted_value = converter(value)
			
			parts_dict[part_uri] = converted_value

		return parts_dict


	def generate_observation(self, observation_uri, target_region_uri, scaling_property_uri, year):
		"""
		Generate an observation for a specific region, property, and year.

		Args:
			observation_uri (str): URI identifying the generated observation.
			target_region_uri (str): URI identifying the observed region.
			scaling_property_uri (str): URI identifying the observed property.
			year (int): Year associated with the observation.

		Returns:
			The generated observation.
		"""

		self.observation_uri = observation_uri
		self.target_region_uri = target_region_uri
		self.scaling_property_uri = scaling_property_uri
		self.temporal_extent = year

		# finding the class name of target feature
		feature_class = self.get_feature_class()

		# fetching the procedure specifications
		self.fetch_scaling_procedure(feature_class)

		parts = self.get_observation_values()

		scaled_value = self.run_scaling_method(self.scaling_method, parts)


		scaled_observation = f'''
		<id/observation/{self.observation_uri}> a sosa:Observation ;  
			sosa:resultTime "2011-10-31T00:00:00Z"^^xsd:dateTime ; 
			sosa:hasFeatureOfInterest {self.target_region_uri} ; 
			sosa:observedProperty {self.scaling_property_uri}; 
			sosa:hasSimpleResult "{scaled_value}"^^xsd:integer ; 
			sosa:usedProcedure sogo:populationUpscaling . '''
		return scaled_observation



In [116]:
sogo = SogoScaling()
obs = sogo.generate_observation(f"{CRISP_BASE_URI}state/1/population", 
                                f"{CRISP_BASE_URI}id/state/1",
                                f"{CRISP_BASE_URI}population",
                                2011)


In [117]:
print(obs)


		<id/observation/http://crispy.ai.wu.ac.at/crisp/state/1/population> a sosa:Observation ;  
			sosa:resultTime "2011-10-31T00:00:00Z"^^xsd:dateTime ; 
			sosa:hasFeatureOfInterest http://crispy.ai.wu.ac.at/crisp/id/state/1 ; 
			sosa:observedProperty http://crispy.ai.wu.ac.at/crisp/population; 
			sosa:hasSimpleResult "285685"^^xsd:integer ; 
			sosa:usedProcedure sogo:populationUpscaling . 
